# Goal: Understand what OMDb returns for a title and estimate data quality before building the enrichment pipeline

In [1]:
# %pip install requests
# %pip install python-dotenv

In [2]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd

In [3]:
load_dotenv('../.env')

True

In [4]:
API_KEY = os.getenv('OMDb_API_KEY')  
BASE_URL = 'https://www.omdbapi.com/'

In [6]:
def omdb_get(params: dict, timeout: int = 20) -> dict:
    if not API_KEY:
        raise ValueError('Set env var OMDB_API_KEY')
    params = {**params, 'apikey': API_KEY, 'r': 'json'}
    r = requests.get(BASE_URL, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

### Raw JSON preview

In [7]:
data = omdb_get({'t': 'Sky Captain and the World of Tomorrow', 'plot': 'short'})
data

{'Title': 'Sky Captain and the World of Tomorrow',
 'Year': '2004',
 'Rated': 'PG',
 'Released': '17 Sep 2004',
 'Runtime': '106 min',
 'Genre': 'Action, Adventure, Mystery',
 'Director': 'Kerry Conran',
 'Writer': 'Kerry Conran',
 'Actors': 'Gwyneth Paltrow, Jude Law, Angelina Jolie',
 'Plot': 'After New York City receives a series of attacks from giant flying robots, a reporter teams up with a pilot in search of their origin, as well as the reason for the disappearances of famous scientists around the world.',
 'Language': 'English, Tibetan, German',
 'Country': 'United Kingdom, Italy, United States',
 'Awards': '8 wins & 19 nominations total',
 'Poster': 'https://m.media-amazon.com/images/M/MV5BMTM0NDQzMDA1NF5BMl5BanBnXkFtZTcwNTU3ODAzMw@@._V1_SX300.jpg',
 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '6.1/10'},
  {'Source': 'Rotten Tomatoes', 'Value': '71%'},
  {'Source': 'Metacritic', 'Value': '64/100'}],
 'Metascore': '64',
 'imdbRating': '6.1',
 'imdbVotes': '90,083'

#### Keys and data types

In [8]:
{k: type(v).__name__ for k, v in data.items()}

{'Title': 'str',
 'Year': 'str',
 'Rated': 'str',
 'Released': 'str',
 'Runtime': 'str',
 'Genre': 'str',
 'Director': 'str',
 'Writer': 'str',
 'Actors': 'str',
 'Plot': 'str',
 'Language': 'str',
 'Country': 'str',
 'Awards': 'str',
 'Poster': 'str',
 'Ratings': 'list',
 'Metascore': 'str',
 'imdbRating': 'str',
 'imdbVotes': 'str',
 'imdbID': 'str',
 'Type': 'str',
 'DVD': 'str',
 'BoxOffice': 'str',
 'Production': 'str',
 'Website': 'str',
 'Response': 'str'}

In [9]:
data.get('Ratings')

[{'Source': 'Internet Movie Database', 'Value': '6.1/10'},
 {'Source': 'Rotten Tomatoes', 'Value': '71%'},
 {'Source': 'Metacritic', 'Value': '64/100'}]

#### Sample of titles

In [10]:
sample_titles = ['Inception', 'The Matrix', 'Avatar', 'Titanic', 'Joker']

rows = []
for t in sample_titles:
    d = omdb_get({'t': t, 'type': 'movie', 'plot': 'short'})
    rows.append({
        'query_title': t,
        'Response': d.get('Response'),
        'Title': d.get('Title'),
        'Year': d.get('Year'),
        'Type': d.get('Type'),
        'imdbID': d.get('imdbID'),
        'Genre': d.get('Genre'),
        'Runtime': d.get('Runtime'),
        'Director': d.get('Director'),
        'Actors': d.get('Actors'),
        'imdbRating': d.get('imdbRating'),
        'imdbVotes': d.get('imdbVotes'),
        'BoxOffice': d.get('BoxOffice'),
        'Awards': d.get('Awards'),
        'Error': d.get('Error')
    })

pd.DataFrame(rows)

,query_title,Response,Title,Year,Type,imdbID,Genre,Runtime,Director,Actors,imdbRating,imdbVotes,BoxOffice,Awards,Error
0,Inception,True,Inception,2010,movie,tt1375666,"Action, Adventure, Sci-Fi",148 min,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...",8.8,"2,781,889","$292,587,330",Won 4 Oscars. 160 wins & 220 nominations total,None
1,The Matrix,True,The Matrix,1999,movie,tt0133093,"Action, Sci-Fi",136 min,"Lana Wachowski, Lilly Wachowski","Keanu Reeves, Laurence Fishburne, Carrie-Anne ...",8.7,"2,228,203","$177,559,005",Won 4 Oscars. 42 wins & 52 nominations total,None
2,Avatar,True,Avatar,2009,movie,tt0499549,"Action, Adventure, Fantasy",162 min,James Cameron,"Sam Worthington, Zoe Saldaña, Sigourney Weaver",7.9,"1,486,308","$785,221,649",Won 3 Oscars. 91 wins & 131 nominations total,None
3,Titanic,True,Titanic,1997,movie,tt0120338,"Drama, Romance",194 min,James Cameron,"Leonardo DiCaprio, Kate Winslet, Billy Zane",8.0,"1,378,313","$674,354,882",Won 11 Oscars. 126 wins & 84 nominations total,None
4,Joker,True,Joker,2019,movie,tt7286456,"Crime, Drama, Thriller",122 min,Todd Phillips,"Joaquin Phoenix, Robert De Niro, Zazie Beetz",8.3,"1,675,451","$335,477,657",Won 2 Oscars. 120 wins & 246 nominations total,None


#### Error cases

In [11]:
omdb_get({'t': 'asdasdasdasd_not_a_movie'})

{'Response': 'False', 'Error': 'Movie not found!'}

### OMDb sample profiling

In [12]:
titles = pd.read_csv('../data/revenues_per_day (1).csv', usecols=['title'])

In [13]:
sample_200_titles = titles.sample(200)['title'].to_list()
len(sample_200_titles)

200

In [14]:
sample_200_titles[:10]

['Kangaroo Jack',
 'The Purge: Anarchy',
 'Johnny English',
 "The Dead Don't Die",
 'The Last Castle',
 'Words and Pictures',
 'Baggage Claim',
 'Journey 2: The Mysterious Island',
 'Alex Rider: Operation Stormbreaker',
 'The Last Kiss']

In [15]:
FIELDS = [
    'Title','Year','Rated','Released','Runtime','Genre','Director','Writer','Actors','Plot',
    'Language','Country','Awards','Poster','Metascore','imdbRating','imdbVotes','imdbID','Type',
    'DVD','BoxOffice','Production','Website'
]

rows = []
for t in sample_200_titles:
    d = omdb_get({'t': t, 'plot': 'short'})
    row = {'query_title': t, 'Response': d.get('Response'), 'Error': d.get('Error')}
    for f in FIELDS:
        row[f] = d.get(f)
    rows.append(row)

df = pd.DataFrame(rows)

In [16]:
df.head()

,query_title,Response,Error,Title,Year,Rated,Released,Runtime,Genre,Director,...,Poster,Metascore,imdbRating,imdbVotes,imdbID,Type,DVD,BoxOffice,Production,Website
0,Kangaroo Jack,True,NaN,Kangaroo Jack,2003,PG,17 Jan 2003,89 min,"Action, Adventure, Comedy",David McNally,...,https://m.media-amazon.com/images/M/MV5BMTk1Nj...,16,4.5,"34,956",tt0257568,movie,N/A,"$66,934,963",N/A,N/A
1,The Purge: Anarchy,True,NaN,The Purge: Anarchy,2014,R,18 Jul 2014,103 min,"Action, Horror, Sci-Fi",James DeMonaco,...,https://m.media-amazon.com/images/M/MV5BMjE2OD...,50,6.4,"171,756",tt2975578,movie,N/A,"$71,962,800",N/A,N/A
2,Johnny English,True,NaN,Johnny English,2003,PG,18 Jul 2003,87 min,"Action, Adventure, Comedy",Peter Howitt,...,https://m.media-amazon.com/images/M/MV5BMTU0MG...,51,6.3,"183,876",tt0274166,movie,N/A,"$28,082,366",N/A,N/A
3,The Dead Don't Die,True,NaN,The Dead Don't Die,2019,R,14 Jun 2019,104 min,"Action, Comedy, Crime",Jim Jarmusch,...,https://m.media-amazon.com/images/M/MV5BMGQzMT...,53,5.4,"101,373",tt8695030,movie,N/A,"$6,563,605",N/A,N/A
4,The Last Castle,True,NaN,The Last Castle,2001,R,19 Oct 2001,131 min,"Action, Drama, Thriller",Rod Lurie,...,https://m.media-amazon.com/images/M/MV5BMTk0Mz...,43,7.0,"90,627",tt0272020,movie,N/A,"$18,244,060",N/A,N/A


#### Response True/False and top errors

In [17]:
print(df['Response'].value_counts(dropna=False))

Response
True     189
False     11
Name: count, dtype: int64
Error
Movie not found!    11
Name: count, dtype: int64


In [30]:
print(df.loc[df['Response'] != 'True', 'Error'].value_counts(dropna=False).head(10))

Error
Movie not found!    11
Name: count, dtype: int64


#### Missing values

In [18]:
missing_mask = df[FIELDS].isna() | (df[FIELDS] == '') | (df[FIELDS] == 'N/A')
missing_pct = (missing_mask.mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    'missing_pct': missing_pct.round(1),
    'missing_cnt': missing_mask.sum().astype(int)
}).sort_values('missing_pct', ascending=False)

missing_summary

,missing_pct,missing_cnt
Website,100.0,200
Production,99.5,199
DVD,99.0,198
Awards,11.5,23
Metascore,11.5,23
Writer,8.5,17
BoxOffice,6.5,13
Rated,6.5,13
Runtime,6.0,12
Director,6.0,12


 #### Which titles fail and why

In [19]:
df_failed = df[df['Response'] != 'True'][['query_title', 'Error']].copy()
df_failed.head(5)

,query_title,Error
8,Alex Rider: Operation Stormbreaker,Movie not found!
34,"Monsters, Inc.2012 3D Release",Movie not found!
37,Avatar2010 Special Edition,Movie not found!
95,Tyler Perry's Madea's Big Happy Family,Movie not found!
104,Poku00e9mon the Movie: The Power of Us,Movie not found!


In [20]:
print(f'Fails amount: {len(df_failed)}')

Fails amount: 11


In [21]:
print(f'Error messages: {df_failed['Error'].unique()}')

Error messages: <StringArray>
['Movie not found!']
Length: 1, dtype: str


#### Ratings profiling and missing ratings

In [22]:
ratings_rows = []
for _, r in df[df['Response'] == 'True'].iterrows():
    d = omdb_get({'i': r['imdbID'], 'plot': 'short'}) if pd.notna(r['imdbID']) else {}
    ratings = d.get('Ratings') or []
    if len(ratings) == 0:
        ratings_rows.append({'imdbID': r['imdbID'], 'source': None, 'value': None})
    else:
        for it in ratings:
            ratings_rows.append({'imdbID': r['imdbID'], 'source': it.get('Source'), 'value': it.get('Value')})

df_ratings = pd.DataFrame(ratings_rows)

print('Movies with no Ratings:', df_ratings['source'].isna().sum())

Movies with no Ratings: 1
source
Internet Movie Database    188
Metacritic                 177
Rotten Tomatoes            177
Name: count, dtype: int64


In [31]:
print(df_ratings['source'].value_counts(dropna=True))

source
Internet Movie Database    188
Metacritic                 177
Rotten Tomatoes            177
Name: count, dtype: int64


In [23]:
df_ratings.head(20)

,imdbID,source,value
0,tt0257568,Internet Movie Database,4.5/10
1,tt0257568,Metacritic,16/100
2,tt2975578,Internet Movie Database,6.4/10
3,tt2975578,Rotten Tomatoes,58%
4,tt2975578,Metacritic,50/100
5,tt0274166,Internet Movie Database,6.3/10
6,tt0274166,Rotten Tomatoes,33%
7,tt0274166,Metacritic,51/100
8,tt8695030,Internet Movie Database,5.4/10
9,tt8695030,Rotten Tomatoes,55%


#### imdbID duplicates

In [24]:
dups = (df[df['Response'] == 'True']
        .groupby('imdbID')['query_title']
        .nunique()
        .sort_values(ascending=False))

print('imdbID with >1 query title:', (dups > 1).sum())
dups[dups > 1].head(20)

imdbID with >1 query title: 0


Series([], Name: query_title, dtype: int64)

#### how many items per list-like field

In [25]:
def count_items(s: pd.Series) -> pd.Series:
    s = s.astype('string')
    s = s.where(~(s.isna() | (s == '') | (s == 'N/A')))
    return s.str.split(', ').map(lambda x: len(x) if isinstance(x, list) else 0)

for col in ['Genre', 'Country', 'Language']:
    c = count_items(df[col])
    print(col, 'min/avg/max:', int(c.min()), round(c.mean(), 2), int(c.max()))

Genre min/avg/max: 0 2.34 3
Country min/avg/max: 0 1.56 6
Language min/avg/max: 0 1.58 6


#### coverage of key analytics fields

In [26]:
ok = df['Response'] == 'True'
key_cols = ['imdbID', 'Genre', 'Country', 'Language', 'imdbRating', 'imdbVotes', 'Metascore', 'Released']

coverage = {}
for c in key_cols:
    miss = df.loc[ok, c].isna() | (df.loc[ok, c] == '') | (df.loc[ok, c] == 'N/A')
    coverage[c] = round((1 - miss.mean()) * 100, 1)

pd.Series(coverage).sort_values(ascending=True)

Metascore      93.7
Released       99.5
imdbRating     99.5
imdbVotes      99.5
Language      100.0
Country       100.0
Genre         100.0
imdbID        100.0
dtype: float64

#### parseability of numeric-like fields

In [27]:
tmp = df.copy()

tmp['runtime_min'] = pd.to_numeric(tmp['Runtime'].astype('string').str.replace(' min', '', regex=False), errors='coerce')
tmp['imdb_rating_num'] = pd.to_numeric(tmp['imdbRating'], errors='coerce')
tmp['metascore_num'] = pd.to_numeric(tmp['Metascore'], errors='coerce')
tmp['imdb_votes_num'] = pd.to_numeric(tmp['imdbVotes'].astype('string').str.replace(',', '', regex=False), errors='coerce')
tmp['boxoffice_num'] = pd.to_numeric(tmp['BoxOffice'].astype('string')
                                     .str.replace('$', '', regex=False)
                                     .str.replace(',', '', regex=False), errors='coerce')

print('Runtime parse failures:', tmp['runtime_min'].isna().mean())
print('imdbRating parse failures:', tmp['imdb_rating_num'].isna().mean())
print('Metascore parse failures:', tmp['metascore_num'].isna().mean())
print('imdbVotes parse failures:', tmp['imdb_votes_num'].isna().mean())
print('BoxOffice parse failures:', tmp['boxoffice_num'].isna().mean())

Runtime parse failures: 0.06
imdbRating parse failures: 0.06
Metascore parse failures: 0.115
imdbVotes parse failures: 0.06
BoxOffice parse failures: 0.065


#### Released date parse and consistency with Year

In [28]:
tmp = df[df['Response'] == 'True'].copy()
tmp['released_dt'] = pd.to_datetime(tmp['Released'], errors='coerce')
tmp['year_num'] = pd.to_numeric(tmp['Year'], errors='coerce')

print('Released parse failures:', tmp['released_dt'].isna().mean())
print('Year parse failures:', tmp['year_num'].isna().mean())

Released parse failures: 0.005291005291005291
Year parse failures: 0.005291005291005291


In [29]:
mismatch = tmp[(~tmp['released_dt'].isna()) & (~tmp['year_num'].isna()) &
               (tmp['released_dt'].dt.year != tmp['year_num'])]
print('Released.year != Year count:', len(mismatch))
mismatch[['query_title','Title','Year','Released']].head(20)

Released.year != Year count: 21


,query_title,Title,Year,Released
5,Words and Pictures,Words and Pictures,2013,17 Jul 2014
11,Song of the Sea,Song of the Sea,2014,20 Nov 2015
23,Jeremiah Tower: The Last Magnificent,Jeremiah Tower: The Last Magnificent,2016,21 Apr 2017
40,The Namesake,The Namesake,2006,06 Apr 2007
44,Thank You for Smoking,Thank You for Smoking,2005,14 Apr 2006
68,The Upside,The Upside,2017,11 Jan 2019
99,The Letters,The Letters,2014,04 Dec 2015
106,Tosca's Kiss,Tosca's Kiss,1984,24 Jul 1985
109,No One Lives,No One Lives,2012,10 May 2013
112,Little Manhattan,Little Manhattan,2005,05 Jan 2006


## OMDb overview 

* What OMDb returns:

  * Most fields come as strings

  * Some fields are comma separated lists stored in a single string (Genre, Country, Language, Actors)

  * Ratings is a nested list of {Source, Value}

  * Missing data is often represented as the literal string 'N/A'


* Sample results

  * Quick sanity check on a few popular titles returned consistent movie metadata and metrics

  * Error format is stable : Response = 'False' and Error explains the reason


* Profiling on 200 random titles from revenues_per_day

  * Most titles were found in OMDb

  * Failures exist and are returned as 'Movie not found!'

  * Some fields are extremely sparse and can be ignored (Website, Production, DVD)

  * Key enrichment fields have good coverage when a title is found

    * imdbID, Genre, Country, Language

    * Released, imdbRating, imdbVotes

    * Metascore is a bit more spotty
   
  * imdbID as the stable movie key for the warehouse

  * Year and Released  are not always the same year


## OMDb enrichment - what we can use 

* Core fields:

  * imdbID - stable movie key for joins and deduplication

  * Title - readable name and a way to validate matches

  * Type - split movies vs series if they appear

  * Year - production year 

  * Released - release date

  * Rated - filtering and rankings by age rating

  * Runtime - bucket movies by length

  * imdbRating - popularity/quality
 
  * imdbVotes - confidence / popularity 

  * Metascore - extra score 

* Extra slicing attributes (Note - these can be stored as raw strings in dim_movie, or normalized into separate dimensions + mapping tables for cleaner filtering and rankings):

  * Genre - rankings and filters by genre 

  * Country - rankings and filters by country 

  * Language - rankings and filters by language 

  * Director - top revenue by director

  * Writer - top revenue by writer

  * Actors - top revenue by actor
 


* Ratings (optional table)

  * movie_ratings - one row per movie per source (IMDb, Rotten Tomatoes, Metacritic)

* Skip on start:

  * Website - usually 'N/A'

  * Production - usually 'N/A'

  * DVD - usually 'N/A'

  * Plot - descriptive only

  * Poster - UI only

  * Awards - unstructured text

  * BoxOffice - total gross, not daily, can be incomplete
